# Абляция представления для recursive `lag_168`

Проверяем, скрывают ли core признаки `weekday`, `hour`, `route_hour`, `hour_weekend`, `is_night` пользу недельного рекурсивного лага. C — основной тест: сохраняет `route`, календарь и погоду. D — диагностический крайний вариант без идентификатора маршрута. A и B воспроизводят ранее проверенные reference и recursive candidate.

Цель — `boardings` для `(route, date, hour)`. Folds: Jan–Apr → May–Jun, Jan–Jun → Jul–Aug, Jan–Aug → Sep–Oct. Параметры tuned LightGBM, обработка категорий, округление и route 5 = 0 заморожены. До начала holdout доступны только прошлые actual; после первой недели лаг берётся из прежних predictions. Фактический target holdout в recursive state не передаётся.

Решение принимаем по temporal WAPE. Gain importance и корреляция прогноза с лагом описывают поведение модели, но не доказывают причинную пользу признака. Если первая неделя лучше, а поздние хуже, это совместимо с накоплением recursive ошибки; здесь сравниваем все три интервала отдельно.

In [1]:
from pathlib import Path
import json, subprocess, sys
import pandas as pd

root = Path.cwd().resolve()
while not (root / 'ml' / 'src' / 'recursive_lag_ablation.py').exists():
    if root == root.parent:
        raise FileNotFoundError('Project root not found')
    root = root.parent
subprocess.run([sys.executable, str(root / 'ml/src/recursive_lag_ablation.py')], cwd=root, check=True)
result = json.loads((root / 'ml/experiments/results/recursive_lag_ablation.json').read_text(encoding='utf-8'))
assert result['reference_A_and_prior_B_reproduced']
result['classification']

'RECURSIVE_MASKING_REJECTED'

In [2]:
rows = []
for name, summary in result['summary'].items():
    rows.append({'variant': name, **{fold['fold']: fold['metrics'][name]['wape'] for fold in result['folds']}, 'pooled_wape': summary['pooled_wape'], 'delta_vs_A': summary['delta_vs_A'], 'delta_vs_B': summary['delta_vs_B'], 'pooled_absolute_error': summary['pooled_absolute_error'], 'wape_score': summary['wape_score']})
display(pd.DataFrame(rows).set_index('variant').round(6))
display(pd.DataFrame({name: {bucket: values[name]['pooled_wape'] for bucket, values in result['pooled_horizons'].items()} for name in result['features']}).round(6))
display(pd.DataFrame({name: {route: value['wape'] for route, value in summary['per_route'].items()} for name, summary in result['summary'].items()}).round(6))

,jan_apr_to_may_jun,jan_jun_to_jul_aug,jan_aug_to_sep_oct,pooled_wape,delta_vs_A,delta_vs_B,pooled_absolute_error,wape_score
variant,,,,,,,,
A,0.141207,0.205422,0.114021,0.150956,0.000000,-0.017251,5287697,0.849044
B,0.189901,0.198844,0.122807,0.168207,0.017251,0.000000,5891961,0.831793
C,0.368503,0.216586,0.192500,0.257934,0.106978,0.089727,9034917,0.742066
D,0.300499,0.204180,0.193396,0.232029,0.081073,0.063822,8127537,0.767971


,A,B,C,D
days_1_7,0.123722,0.179136,0.186333,0.191298
days_8_28,0.153421,0.169625,0.231909,0.222468
days_29_61,0.154648,0.164822,0.288981,0.245408


,A,B,C,D
1,0.124881,0.152070,0.224151,0.182704
5,NaN,NaN,NaN,NaN
7,0.271423,0.279044,0.391477,0.298466
11,0.126133,0.141426,0.236031,0.181381
12,0.121498,0.139940,0.225737,0.161526
17,0.101852,0.115630,0.208383,0.247909
25,0.193525,0.215761,0.347791,0.374495
26,0.165674,0.178832,0.239645,0.225456
28,0.184384,0.213251,0.281525,0.300858
50,0.212339,0.236652,0.328324,0.281365


In [3]:
diagnostics = []
for fold in result['folds']:
    for name, values in fold['diagnostics'].items():
        diagnostics.append({'fold': fold['fold'], 'variant': name, **values['importance'], 'prediction_lag_correlation': values['validation_lag']['prediction_lag_168_correlation']})
display(pd.DataFrame(diagnostics).round(4))

,fold,variant,lag_168_gain,rank,feature_count,total_gain_fraction,prediction_lag_correlation
0,jan_apr_to_may_jun,B,5.087020e+05,2,13,0.2220,0.9963
1,jan_apr_to_may_jun,C,1.432857e+06,1,8,0.7647,0.9715
2,jan_apr_to_may_jun,D,1.373123e+06,1,7,0.7639,0.9904
3,jan_jun_to_jul_aug,B,7.586560e+05,2,13,0.2196,0.9992
4,jan_jun_to_jul_aug,C,2.318775e+06,1,8,0.8232,0.9991
5,jan_jun_to_jul_aug,D,2.202980e+06,1,7,0.8085,0.9995
6,jan_aug_to_sep_oct,B,3.111691e+06,1,13,0.7571,0.9990
7,jan_aug_to_sep_oct,C,3.308944e+06,1,8,0.8603,0.9992
8,jan_aug_to_sep_oct,D,3.195728e+06,1,7,0.8532,0.9992


## Вывод

`RECURSIVE_MASKING_REJECTED`: C существенно хуже B на всех трёх folds и на каждом горизонте. Лаг получает больше gain после удаления core признаков, однако temporal WAPE резко растёт. На днях 1–7 C тоже хуже B; после 29-го дня разрыв увеличивается, что согласуется с recursive propagation. D лучше C по pooled WAPE, поэтому отсутствие `route` нельзя считать единственным объяснением провала C; D всё равно намного хуже A/B. Это сравнительная диагностика, а не причинное доказательство механизма. Accepted registry и primary v4 не изменяются; final Jan–Oct fit и submission не выполняются.